In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from math import nan
import pandas as pd
import yaml
import xesmf as xe
import dask
import socket
from pathlib import Path
import sys
from datetime import datetime

sys.path.append("/glade/u/home/islas/python/CESM2_with_CMIP7_aer/utils/")
from reading_utils import *
from convert_units import *
from spread_emissions_func import *
from molecular_weights import *

#import importlib
#import molecular_weights as molecular_weights
#importlib.reload(molecular_weights)

import os
import csv

import psutil
import os

### Loop over species and make the files

In [2]:
# Set up species to loop over and the emissions type and the forcing type
#process_species = ['bc_a4','pom_a4','SO2','so4_a1','SOAG']
process_species = ['SOAG']
emiss_type = "sfc"
forcing_type = "bmb"

for species in process_species:

    print(species)
    
    # Set up output path and log file
    outpath="/glade/campaign/cgd/cas/islas/python_savs/CESM2_with_CMIP7_aer/make_emissions/"+forcing_type+"/"+emiss_type+"/CMIP7/"
    os.makedirs(outpath, exist_ok=True)
    os.makedirs(outpath+'/DIAGS/', exist_ok=True)
    logfilename = outpath+'/DIAGS/'+species+'_log.csv'
    if os.path.exists(logfilename):
        os.remove(logfilename)
    

    # open log file for writing
    file = open(logfilename, mode='w', newline='')
    writer = csv.writer(file)

    # Set up output filename
    fout = outpath+"/"+species+"_"+forcing_type+"_"+emiss_type+"_cmip7_for_cesm2_negspread.nc"

    # Set up spreading diagnostic filename
    fout_diags = outpath+"/DIAGS/DIAGS_"+species+"_"+forcing_type+"_"+emiss_type+"_cmip7_for_cesm2_negspread_diags.nc"

    # Read in the cmip6 piControl, cmip7 piControl, cmip6 historical and cmip7 historical
    cmip6_pi = read_emissions('../../../../emission_file_'+forcing_type+'_'+emiss_type+'_info.yml',species,'cmip6','piControl')
    cmip7_pi = read_emissions('../../../../emission_file_'+forcing_type+'_'+emiss_type+'_info.yml',species,'cmip7','piControl', dstgrid = 'f09')
    cmip6_hist = read_emissions('../../../../emission_file_'+forcing_type+'_'+emiss_type+'_info.yml',species,'cmip6','hist')
    cmip7_hist = read_emissions('../../../../emission_file_'+forcing_type+'_'+emiss_type+'_info.yml',species,'cmip7','hist', dstgrid='f09')

    # save the attributes for CMIP7
    cmip7_attrs = cmip7_hist.attrs
    
    # Enforce the same longitudes and latitudes for all arrays
    cmip7_pi['lon'] = cmip6_pi.lon ; cmip7_pi['lat'] = cmip6_pi.lat
    cmip6_hist['lon'] = cmip6_pi.lon ; cmip6_hist['lat'] = cmip6_pi.lat
    cmip7_hist['lon'] = cmip6_pi.lon ; cmip7_hist['lat'] = cmip6_pi.lat

    # Sum up the different emission types
    cmip6_pi = cmip6_pi.sum('emiss_type')
    cmip7_pi = cmip7_pi.sum('emiss_type')
    cmip6_hist = cmip6_hist.sum('emiss_type')
    cmip7_hist = cmip7_hist.sum('emiss_type')

    # Add the CMIP7 anomalies onto the CMIP6 piControl
    cmip7_1850 = cmip7_pi.groupby('time.month').mean('time')
    cmip6_1850 = cmip6_pi.groupby('time.month').mean('time')
    cmip7_anom = cmip7_hist.groupby('time.month') - cmip7_1850
    cmip7_for_cesm2 = cmip6_1850 + cmip7_anom.groupby('time.month')

    # Ditch pre-1848
    # keeping 1848 and 1849 to compute the running mean as in the CMIP7 smoothed dataset
    cmip7_for_cesm2 = cmip7_for_cesm2.sel(time=slice("1848-01-01","2100-12-31"))

    # Spread the negatives and write the outpu
    nring=60
    output_spread_by_ring_tg = []

    process = psutil.Process(os.getpid())
    writer.writerow(['time','lon','lat','deficit'])
    lon = cmip7_for_cesm2.lon.values
    lat = cmip7_for_cesm2.lat.values
    fixed = np.empty( cmip7_for_cesm2.shape, dtype = cmip7_for_cesm2.dtype)
    for itime in np.arange(0,cmip7_for_cesm2.time.size):

        this_time = cmip7_for_cesm2.isel(time=itime).load()

        if itime % 120 == 0:
            print(this_time.time.values)

        newdat = this_time.values.copy()
        ilat_neg, ilon_neg = np.where( newdat < 0)

        # Note "time" is really "ring" here
        spread_by_ring = xr.DataArray(np.zeros((nring, len(lat), len(lon)), dtype=np.float64), dims=['time','lat','lon'],
                                  coords=[np.arange(0,60,1), cmip7_for_cesm2.lat, cmip7_for_cesm2.lon])
    
        for ilat, ilon in zip(ilat_neg, ilon_neg):
            writer.writerow([this_time.time.values, lon[ilon], lat[ilat], newdat[ilat,ilon]])
            newdat, this_spread = spread_negative_emissions_np(newdat,lon,lat,ilon,ilat)
            spread_by_ring[:,ilat,ilon] += this_spread

        # Convert spread by ring to Tg
        spread_by_ring_tg = convert_molecules_to_tg_specifywgt(spread_by_ring, mol_weights[species])
        spread_by_ring_tg = spread_by_ring_tg.rename({'time':'ring'})
        output_spread_by_ring_tg.append(spread_by_ring_tg)

        fixed[itime,:,:] = newdat
    
    fixed = xr.DataArray(fixed, dims=cmip7_for_cesm2.dims, coords = cmip7_for_cesm2.coords, name='emiss')
    fixed.attrs = cmip7_attrs
    fixed.attrs["description"] = "CMIP7 emission anomalies added onto the CMIP6 piControl, adjusted to spread negative values"
    fixed.attrs["method"] = "Negative emissions redistributed to neighboring grid cells"

    # add date variable
    date = [str(itime.dt.year.values).zfill(4)+str(itime.dt.month.values).zfill(2)+str(itime.dt.day.values).zfill(2) for itime in fixed.time ]
    date = xr.DataArray(date, dims=['time'], coords=[fixed.time], name='date')
    date.attrs['units'] = 'YYYYMMDD'
    date.attrs['long_name'] = 'Date'
    
    datout = xr.merge([fixed, date])
    datout.attrs.update({
        "author": "Isla Simpson (islas@ucar.edu)",
        "creation_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "creation_script": os.path.basename("/glade/u/home/islas/python/CESM2_with_CMIP7_aer/make_emissions/vnv/sfc/CMIP7/make_bmb_emissions.ipynb")})
    
    datout.to_netcdf(fout)

    output_spread_by_ring_tg = xr.concat(output_spread_by_ring_tg, dim='time')
    output_spread_by_ring_tg['time'] = fixed.time.values
    output_spread_by_ring_tg.to_netcdf(fout_diags)
    file.close()    

SOAG
/glade/p/cesmdata/cseg/inputdata/atm/cam/chem/emis/emissions_ssp370-BB_smoothed/emissions-cmip6-ScenarioMIP_IAMC-AIM-ssp370-1-1_smoothed_SOAGx1.5_bb_surface_mol_175001-210101_0.9x1.25_c20201016.nc
emiss_bb
/glade/campaign/cgd/cas/islas/python_savs/CESM2_with_CMIP7_aer/make_emissions/REMAP/f09/SOAGx1.5_bmb_sfc_from_DRES-CMIP-BB4CMIP7-2-1_20251102.nc
emiss
/glade/p/cesmdata/cseg/inputdata/atm/cam/chem/emis/emissions_ssp370-BB_smoothed/emissions-cmip6-ScenarioMIP_IAMC-AIM-ssp370-1-1_smoothed_SOAGx1.5_bb_surface_mol_175001-210101_0.9x1.25_c20201016.nc
emiss_bb
/glade/campaign/cgd/cas/islas/python_savs/CESM2_with_CMIP7_aer/make_emissions/REMAP/f09/SOAGx1.5_bmb_sfc_from_DRES-CMIP-BB4CMIP7-2-1_20251102.nc
emiss
1848-01-16 12:00:00
1858-01-16 12:00:00
1868-01-16 12:00:00
1878-01-16 12:00:00
1888-01-16 12:00:00
1898-01-16 12:00:00
1908-01-16 12:00:00
1918-01-16 12:00:00
1928-01-16 12:00:00
1938-01-16 12:00:00
1948-01-16 12:00:00
1958-01-16 12:00:00
1968-01-16 12:00:00
1978-01-16 12:00:00
1